In [16]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

In [17]:
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

In [18]:
BASE_URL = "https://www.globalfirepower.com/countries-listing.php"

OTHER_SOURCES = {
    'https://www.globalfirepower.com/total-population-by-country.php': 'total_population',
    'https://www.globalfirepower.com/available-military-manpower.php': 'total_military_manpower',
    'https://www.globalfirepower.com/manpower-fit-for-military-service.php': 'fit_for_service',
    'https://www.globalfirepower.com/manpower-reaching-military-age-annually.php': 'population_reaching_military_age_annually',
    'https://www.globalfirepower.com/active-military-manpower.php': 'active_personnel',
    'https://www.globalfirepower.com/active-reserve-military-manpower.php': 'reserve_personnel',
    'https://www.globalfirepower.com/manpower-paramilitary.php': 'paramilitary',
    'https://www.globalfirepower.com/aircraft-total.php': 'total_military_aircraft',
    'https://www.globalfirepower.com/aircraft-total-fighters.php': 'fighter_aircraft',
    'https://www.globalfirepower.com/aircraft-total-attack-types.php': 'attack_aircraft',
    'https://www.globalfirepower.com/aircraft-total-transports.php': 'transport_aircraft',
    'https://www.globalfirepower.com/aircraft-total-trainers.php': 'trainer_aircraft',
    'https://www.globalfirepower.com/aircraft-total-special-mission.php': 'special_mission_aircraft',
    'https://www.globalfirepower.com/aircraft-total-tanker-fleet.php': 'tanker_aircraft',
    'https://www.globalfirepower.com/aircraft-helicopters-total.php': 'total_military_helicopters',
    'https://www.globalfirepower.com/aircraft-helicopters-attack.php': 'attack_helicopters',
    'https://www.globalfirepower.com/armor-tanks-total.php': 'tanks',
    'https://www.globalfirepower.com/armor-apc-total.php': 'armored_fighting_vehicles',
    'https://www.globalfirepower.com/armor-self-propelled-guns-total.php': 'self_propelled_artillery',
    'https://www.globalfirepower.com/armor-towed-artillery-total.php': 'towed_artillery',
    'https://www.globalfirepower.com/armor-mlrs-total.php': 'rocket_projectors',
    'https://www.globalfirepower.com/navy-ships.php': 'total_naval_fleet',
    'https://www.globalfirepower.com/navy-force-by-tonnage.php': 'total_naval_fleet_tonnage_mt',
    'https://www.globalfirepower.com/navy-aircraft-carriers.php': 'aircraft_carriers',
    'https://www.globalfirepower.com/navy-helo-carriers.php': 'helicopter_carriers',
    'https://www.globalfirepower.com/navy-submarines.php': 'submarines',
    'https://www.globalfirepower.com/navy-destroyers.php': 'destroyers',
    'https://www.globalfirepower.com/navy-frigates.php': 'frigates',
    'https://www.globalfirepower.com/navy-corvettes.php': 'corvettes',
    'https://www.globalfirepower.com/navy-patrol-coastal-craft.php': 'coastal_patrol_craft',
    'https://www.globalfirepower.com/navy-mine-warfare-craft.php': 'mine_warfare_craft',
    'https://www.globalfirepower.com/defense-spending-budget.php': 'defense_budget_usd',
    'https://www.globalfirepower.com/external-debt-by-country.php': 'external_debt_usd',
    'https://www.globalfirepower.com/purchasing-power-parity.php': 'purchasing_power_parity_usd',
    'https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php': 'foreign_exchange_and_gold_reserves_usd',
    'https://www.globalfirepower.com/major-serviceable-airports-by-country.php': 'total_serviceable_airports',
    'https://www.globalfirepower.com/labor-force-by-country.php': 'labour_force',
    'https://www.globalfirepower.com/major-ports-and-terminals.php': 'major_ports_and_terminals',
    'https://www.globalfirepower.com/merchant-marine-strength-by-country.php': 'total_merchant_marine_fleet',
    'https://www.globalfirepower.com/railway-coverage.php': 'railway_coverage_km',
    'https://www.globalfirepower.com/roadway-coverage.php': 'roadway_coverage_km',
    'https://www.globalfirepower.com/oil-production-by-country.php': 'oil_production_bbl',
    'https://www.globalfirepower.com/oil-consumption-by-country.php': 'oil_consumption_bbl',
    'https://www.globalfirepower.com/proven-oil-reserves-by-country.php': 'proven_oil_reserves_bbl',
    'https://www.globalfirepower.com/natural-gas-production-by-country.php': 'natural_gas_production_cum',
    'https://www.globalfirepower.com/natural-gas-consumption-by-country.php': 'natural_gas_consumption_cum',
    'https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php': 'proven_natural_gas_reserves_cum',
    'https://www.globalfirepower.com/coal-production-by-country.php': 'coal_production_cum',
    'https://www.globalfirepower.com/coal-consumption-by-country.php': 'coal_consumption_mt',
    'https://www.globalfirepower.com/proven-coal-reserves-by-country.php': 'proven_coal_reserves_cum',
    'https://www.globalfirepower.com/square-land-area.php': 'total_land_area_sq_km',
    'https://www.globalfirepower.com/coastline-coverage.php': 'coastline_coverage_km',
    'https://www.globalfirepower.com/border-coverage.php': 'border_coverage_km',
    'https://www.globalfirepower.com/waterway-coverage.php': 'waterway_coverage_km'
}

In [19]:
def scrape_page(url, column_name):
    """
    Scrapes military and related statistics for all countries from a Global Firepower
    ranking page and returns the extracted data as a pandas DataFrame.

    The function sends an HTTP GET request to the provided URL, parses the HTML
    content using BeautifulSoup, extracts each country's name and its corresponding
    metric value from the page's `recordsetContainer` elements, and stores the
    results in a DataFrame.

    Parameters
    ----------
    url : str
        The URL of the Global Firepower page containing the ranking data.

    column_name : str
        The name of the metric to be used as the column header in the returned
        DataFrame (e.g., 'total_population', 'fighter_aircraft').

    Returns
    -------
    pandas.DataFrame
        A DataFrame with two columns:
        - 'country' : Name of the country.
        - column_name : Corresponding metric value extracted from the page.

    Notes
    -----
    - Assumes that each country's data is contained within a
      `recordsetContainer` HTML element.
    - Country names are extracted from the `longFormName` element, with
      `shortFormName` used as a fallback if necessary.
    - The returned DataFrame can be merged with other DataFrames on the
      'country' column to create a consolidated dataset.
    """
    print(f"Scraping -> {column_name}")

    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.find_all("div", class_="recordsetContainer")

    rows = []

    for card in cards:

        country = ""

        long_name = card.find("div", class_="longFormName")
        if long_name:
            country = long_name.get_text(" ", strip=True)

        if not country:
            short_name = card.find("div", class_="shortFormName")
            if short_name:
                country = short_name.get_text(" ", strip=True)

        value = ""

        value_div = card.find("div", class_="valueContainer")
        if value_div:
            value = value_div.get_text(" ", strip=True)

        rows.append([country, value])

    df = pd.DataFrame(rows, columns=["country", column_name])

    return df


master_df = None

for url, column in OTHER_SOURCES.items():

    df = scrape_page(url, column)

    if master_df is None:
        master_df = df
    else:
        master_df = master_df.merge(df, on="country", how="outer")

    time.sleep(1)

master_df.to_csv("military_raw_data.csv", index=False)

Scraping -> total_population
Scraping -> total_military_manpower
Scraping -> fit_for_service
Scraping -> population_reaching_military_age_annually
Scraping -> active_personnel
Scraping -> reserve_personnel
Scraping -> paramilitary
Scraping -> total_military_aircraft
Scraping -> fighter_aircraft
Scraping -> attack_aircraft
Scraping -> transport_aircraft
Scraping -> trainer_aircraft
Scraping -> special_mission_aircraft
Scraping -> tanker_aircraft
Scraping -> total_military_helicopters
Scraping -> attack_helicopters
Scraping -> tanks
Scraping -> armored_fighting_vehicles
Scraping -> self_propelled_artillery
Scraping -> towed_artillery
Scraping -> rocket_projectors
Scraping -> total_naval_fleet
Scraping -> total_naval_fleet_tonnage_mt
Scraping -> aircraft_carriers
Scraping -> helicopter_carriers
Scraping -> submarines
Scraping -> destroyers
Scraping -> frigates
Scraping -> corvettes
Scraping -> coastal_patrol_craft
Scraping -> mine_warfare_craft
Scraping -> defense_budget_usd
Scraping -> e

In [20]:
print(master_df.head())

       country total_population total_military_manpower fit_for_service  \
0  Afghanistan       40,121,552              15,647,405       8,826,741   
1      Albania        3,107,100               1,522,479       1,292,554   
2      Algeria       47,022,473              22,570,787      19,185,169   
3       Angola       37,202,061               7,440,412       3,720,206   
4    Argentina       46,994,384              20,677,529      17,575,900   

  population_reaching_military_age_annually active_personnel  \
0                                   842,553           75,000   
1                                    62,142            7,500   
2                                   752,360          130,000   
3                                   372,021          107,000   
4                                   704,916          108,000   

  reserve_personnel paramilitary total_military_aircraft fighter_aircraft  \
0                 0       90,000                       5                0   
1         

In [21]:
print(master_df.shape)

(145, 55)
